In [1]:
%load_ext autoreload

In [2]:
%autoreload 2

In [3]:
import os
import sys
import numpy as np

module_path = os.path.abspath(os.path.join('..', ''))
if module_path not in sys.path:
    sys.path.append(module_path)

from rect_prune_ood.lib.util.paths import get_top_level_path

imagenet_c_pertubations = [
        "brightness",
        "defocus_blur",
        "fog",
        "gaussian_blur",
        "glass_blur",
        "jpeg_compression",
        "pixelate",
        "shot_noise",
        "spatter",
        "zoom_blur",
        "contrast",
        "elastic_transform",
        "frost",
        "gaussian_noise", 
        "impulse_noise",
        "motion_blur",
        "saturate",
        "snow",
        "speckle_noise"
    ]

all_activations = []

final_data = None

results_dir = os.path.join(get_top_level_path(), 'results', 'imagenet', "baseline", "imagenet_c")
output_file = os.path.join(results_dir, "imagenet_c_activations.npz")

# Preallocate final array (16, 2048, 5, 50000) assuming 16 perturbations
final_data = np.empty((len(imagenet_c_pertubations), 2048, 5, 50000), dtype=np.float32)

for idx, perturbation in enumerate(imagenet_c_pertubations):
    print(f"Loading {perturbation} activations")

    # Load activation file with memory mapping
    activations = np.load(os.path.join(results_dir, f"imagenet_c_{perturbation}_activations.npy"),
                          allow_pickle=True, mmap_mode='r')

    # Reshape and transpose
    activations = activations.reshape(5, 50000, 2048).transpose(2, 0, 1)  # (2048, 5, 50000)

    # Store directly into preallocated array
    final_data[idx] = activations  # No concatenation!

    print(f"Stored {perturbation}, current shape: {final_data.shape}")

# Save final merged array in a compressed format
print("Final merge complete! Shape:", final_data.shape)  # Should be (16, 2048, 5, 50000)
print("Saving to:", output_file)
np.save(os.path.join(results_dir, "imagenet_c_activations.npy"), final_data)
#np.savez_compressed(output_file, activations=final_data)

# for idx, perturbation in enumerate(imagenet_c_pertubations):
#     print(f"Loading {perturbation} activations")

#     # Load activation file
#     activations = np.load(os.path.join(results_dir, f"imagenet_c_{perturbation}_activations.npy"), allow_pickle=True, mmap_mode='r')

#     # Reshape and transpose
#     #activations = activations.reshape(5, 50000, 2048).transpose(2, 0, 1)  # Shape: (2048, 5, 50000)

#     # Expand dimensions to match the final shape (1, 2048, 5, 50000)
#     activations = np.expand_dims(activations, axis=0)  # Shape: (1, 2048, 5, 50000)

#     # If first iteration, initialize final_data
#     if final_data is None:
#         final_data = activations
#     else:
#         # Concatenate along the first dimension
#         final_data = np.concatenate((final_data, activations), axis=0)

#     print(f"Loaded {perturbation}, current shape: {final_data.shape}")

# # Save final merged array
# np.save(os.path.join(results_dir, "imagenet_c_activations.npy"), final_data)

# print("Final merge complete! Shape:", final_data.shape)  # Should be (16, 2048, 5, 50000)


Loading brightness activations
Stored brightness, current shape: (19, 2048, 5, 50000)
Loading defocus_blur activations
Stored defocus_blur, current shape: (19, 2048, 5, 50000)
Loading fog activations
Stored fog, current shape: (19, 2048, 5, 50000)
Loading gaussian_blur activations
Stored gaussian_blur, current shape: (19, 2048, 5, 50000)
Loading glass_blur activations
Stored glass_blur, current shape: (19, 2048, 5, 50000)
Loading jpeg_compression activations
Stored jpeg_compression, current shape: (19, 2048, 5, 50000)
Loading pixelate activations
Stored pixelate, current shape: (19, 2048, 5, 50000)
Loading shot_noise activations
Stored shot_noise, current shape: (19, 2048, 5, 50000)
Loading spatter activations
Stored spatter, current shape: (19, 2048, 5, 50000)
Loading zoom_blur activations
Stored zoom_blur, current shape: (19, 2048, 5, 50000)
Loading contrast activations
Stored contrast, current shape: (19, 2048, 5, 50000)
Loading elastic_transform activations
Stored elastic_transform

In [ ]:
import os
import sys
import numpy as np

module_path = os.path.abspath(os.path.join('..', ''))
if module_path not in sys.path:
    sys.path.append(module_path)

from rect_prune_ood.lib.util.paths import get_top_level_path

results_dir = os.path.join(get_top_level_path(), 'results', 'imagenet', "baseline", "imagenet_c")

data = np.load(os.path.join(results_dir, "imagenet_c_activations.npy"))

# save severity 5 activations in a separate file
#np.save(os.path.join(results_dir, "imagenet_c_activations_severity_3.npy"), data[:, :, 2, :])
np.save(os.path.join('/fastdata', 'vilab17', "imagenet_c_activations_severity_3.npy"), data[:, :, 2, :])

: 

In [4]:
data = np.load(os.path.join(results_dir, "imagenet_c_activations_severity_5.npy"))

print(data.shape)

(19, 2048, 50000)


In [13]:
import torchvision.models as models

# Load MobileNetV2
model = models.mobilenet_v2(pretrained=True)

# Get all named layers in a list
layers = list(model.named_modules())

# Print the second-to-last layer
second_last_layer_name, second_last_layer = layers[-2]

print(f'Second-to-last layer name: {second_last_layer_name}')
print(f'Second-to-last layer: {second_last_layer}')

# Print the second-to-last layer
# second_last_layer_name, second_last_layer = layers[-1]

# print(f"Second-to-last layer name: {second_last_layer_name}")
# print(f"Second-to-last layer: {second_last_layer}")

print(model.__dict__['_modules']['Conv2dNormActivation.0'])

Second-to-last layer name: classifier.0
Second-to-last layer: Dropout(p=0.2, inplace=False)


/visinf/home/vilab18/miniforge3/envs/rectPruneEnv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/visinf/home/vilab18/miniforge3/envs/rectPruneEnv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


KeyError: 'Conv2dNormActivation.0'

In [ ]:
import os

import numpy as np

from rect_prune_ood.lib.util.paths import get_top_level_path

# This enables imports like
# from rect_prune_ood.lib.prunging import X

In [27]:
from matplotlib import pyplot as plt


def activation_plot(activations, y_max, y_min, name, col):
	plt.bar(np.arange(2048), activations, width=1.0, color=col)
	plt.ylim(y_min, y_max)
	# mean_activation_means = np.mean(activations)
	# plt.axhline(mean_activation_means, color='black', linestyle='--', label=f'Mean: {mean_activation_means:.2f}')
	plt.xticks([])
	plt.yticks([])
	plt.xlabel('Channels')
	plt.ylabel('Activation Variance')
	plt.savefig(f'{name}.png')

	plt.clf()

In [28]:
results_dir = os.path.join(
	get_top_level_path(), 'results', 'imagenet', 'baseline_no_pruning'
)

activations = np.load(
	os.path.join(results_dir, 'imagenet_activations.npy'), allow_pickle=True
)

activations_perturbed = np.load(
	os.path.join(results_dir, 'imagenet_perturbed_activations.npy'), allow_pickle=True
)

activation_means = np.var(activations, axis=0).reshape(1, 2048)
print(activation_means.shape)

perturbed_activation_means = np.var(
	activations_perturbed.reshape(4, 50000, 2048), axis=1
)
print(perturbed_activation_means.shape)

y_max = max(np.max(activation_means), np.max(perturbed_activation_means))
y_min = min(np.min(activation_means), np.min(perturbed_activation_means))


names = [
	'normal_activastions',
	'ood_activations0',
	'ood_activations1',
	'ood_activations2',
	'ood_activations3',
]
colors = ['b', 'r', 'g', 'y', 'c']
all_activations = np.concatenate((activation_means, perturbed_activation_means), axis=0)

for i, activations in enumerate(all_activations):
	activation_plot(activations, y_max, y_min, names[i], colors[i])

(1, 2048)
(4, 2048)


<Figure size 640x480 with 0 Axes>

In [6]:
results_dir = os.path.join(get_top_level_path(), 'results', 'imagenet', "baseline")

activations = np.load(
	os.path.join(results_dir, 'imagenet_activations.npy'), allow_pickle=True
)

activations.shape

activations_perturbed = np.load(
	os.path.join(results_dir, 'imagenet_perturbed_activations.npy'), allow_pickle=True
)

print(activations_perturbed.shape)

all_activations = result = np.concatenate((activations, activations_perturbed), axis=0)

(200000, 2048)


In [7]:
def imagewise_variance(normal_activations, perturbed_activations, p=90, num_augments=4):
	assert perturbed_activations.shape[0] % num_augments == 0
	# group perturbed activations by image
	grouped_perturbed = perturbed_activations.reshape(
		normal_activations.shape[0], num_augments, normal_activations.shape[1]
	)
	# take mean of imagewise activations to get rough activation level on perturbed images in general
	# maybe this step can be skipped for experimentiation
	mean_imagewise_perturbed = grouped_perturbed.mean(axis=1)

	# concatenate perturbed and normal acticvations
	all_activations = np.stack((normal_activations, mean_imagewise_perturbed), axis=1)

	iamgewise_variance = np.var(all_activations, axis=1)

	max_indices = np.argmax(iamgewise_variance, axis=1)

	index_counts = np.bincount(max_indices, minlength=2048)
	ranked_indices = np.argsort(index_counts)[::-1]

	top_10th_count = int(len(ranked_indices) * 0.1)
	ranked_indices = ranked_indices[top_10th_count:]

	return ranked_indices

In [8]:
imagewise_variance(activations, activations_perturbed)

array([   0,    0,    0, ...,  541, 1688, 1690])

In [6]:
all_activations.shape

results_dir2 = os.path.join(get_top_level_path(), 'results', 'masks')
mask = np.load(
	os.path.join(results_dir2, 'prune_top_10_id_idperturbed_variance.npy'),
	allow_pickle=True,
)

variance = np.std(all_activations, axis=0)

# Calculate the 10th percentile
percentile_90 = np.percentile(variance, 90)

# Get the indices of values in the top 10%
top_10_percent_indices = np.where(variance <= percentile_90)[0]

assert mask.all() == top_10_percent_indices.all()

In [7]:
# excluding top 10 highes activation variances

variance = np.std(all_activations, axis=0)

print(variance.shape)

# Calculate the 10th percentile
percentile_90 = np.percentile(variance, 90)

# Get the indices of values in the top 10%
top_10_percent_indices = np.where(variance <= percentile_90)[0]

print(top_10_percent_indices)

print((2048 * 2047 / 2), top_10_percent_indices.sum())

#np.save("prune_top_10_id_idperturbed_variance.npy", top_10_percent_indices)

(2048,)
[   0    1    2 ... 2045 2046 2047]
2096128.0 1890338


In [10]:
variance = np.std(activations, axis=0)
variance_percentile = np.percentile(variance, 90)

print(variance_percentile)
print(variance.shape)
print(variance.mean())
print(variance.max(), variance.min())

0.5605087
(2048,)
0.44736767
1.1170037 0.112264246


In [ ]:
# Create the bar plot
plt.figure(figsize=(12, 6))
plt.bar(range(len(variance)), variance, width=1.0, color='blue', edgecolor='none')
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('Bar Plot of 2048 Values')
plt.show()